# EMS Optimization - Policy Generation and Comparison

**Phase 3: Allocation Policy Optimization**

This notebook documents the optimization-based allocation policies and compares their performance across different unit counts.

## Overview

We generate and compare five allocation policies:
- **P0**: Uniform baseline (equal distribution)
- **P1**: Demand-proportional (allocate based on demand served)
- **P2**: Demand-weighted optimized (minimize demand-weighted response time)
- **P2b**: P-Median optimized (minimize total response time)
- **P2c**: Maximal coverage (maximize demand covered within 8 minutes)

We evaluate performance for K ∈ {20, 30, 40, 48} units.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

BASE_DIR = Path('/home/ubuntu/ems-optimization')
RESULTS_DIR = BASE_DIR / 'results' / 'optimization'

# Load results
results = pd.read_csv(RESULTS_DIR / 'policy_comparison.csv')
sensitivity = pd.read_csv(RESULTS_DIR / 'sensitivity_analysis.csv', header=[0,1], index_col=0)

with open(RESULTS_DIR / 'findings_summary.json') as f:
    findings = json.load(f)

print(f"Loaded {len(results)} optimization results")

## 1. Policy Comparison at K=40

Representative case with 40 units (current Manhattan EMS capacity).

In [ ]:
K40 = results[results['K'] == 40].copy()
K40_display = K40[[
    'policy_name', 'response_time', 'pct_demand_covered', 
    'num_firehouses_used', 'max_units_at_firehouse', 'solve_time_sec'
]]

print("\nPerformance at K=40:")
print(K40_display.to_string(index=False))

# Improvement over baseline
P0_rt = K40[K40['policy_id'] == 'P0']['response_time'].values[0]
P2_rt = K40[K40['policy_id'] == 'P2']['response_time'].values[0]
improvement = (P0_rt - P2_rt) / P0_rt * 100

print(f"\nP2 improvement over P0: {improvement:.1f}% faster response time")

### Key Findings at K=40:

1. **All policies achieve 100% coverage** at K=40
2. **Optimized policies (P2, P2b) are identical** - both achieve 2.49 min response time
3. **P2 is 17% faster** than uniform (P0) allocation
4. **P2 uses fewer firehouses** (24) than P2b (40), achieving same performance
5. **Maximal coverage (P2c)** trades off response time for sparse allocation (only 10 firehouses)

## 2. Allocation Patterns

How units are distributed across firehouses.

In [ ]:
# Load K=40 allocations
alloc_K40 = pd.read_csv(RESULTS_DIR / 'allocations_K40.csv', index_col=0)

print("\nAllocation Statistics (K=40):")
print(alloc_K40.describe().round(2))

print("\nFirehouses with max allocation (5 units):")
for policy in ['P0', 'P1', 'P2', 'P2b', 'P2c']:
    max_allocated = (alloc_K40[policy] == 5).sum()
    print(f"  {policy}: {max_allocated} firehouses")

print("\nSample allocations (top 10 by P2):")
top_10 = alloc_K40.nlargest(10, 'P2')
print(top_10.to_string())

### Allocation Patterns:

- **P0 (Uniform)**: Every firehouse gets exactly 1 unit (40 firehouses)
- **P1 (Demand-Proportional)**: Concentrates units in high-demand areas (up to 4 units)
- **P2 (Demand-Weighted)**: Most concentrated - several firehouses get max allocation (5 units)
- **P2b (P-Median)**: Spreads units across all firehouses (1 each)
- **P2c (Maximal Coverage)**: Extreme concentration - only 10 firehouses with 5 units each

## 3. Sensitivity to Unit Count (K)

How performance changes as we vary the number of available units.

In [ ]:
print("\nSensitivity Analysis:")
print(sensitivity)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

policies = ['P0', 'P1', 'P2', 'P2b', 'P2c']
K_values = [20, 30, 40, 48]

# Response time
ax = axes[0]
for policy in policies:
    data = results[results['policy_id'] == policy]
    ax.plot(data['K'], data['response_time'], 'o-', label=policy, linewidth=2, markersize=8)
ax.set_xlabel('Number of Units (K)')
ax.set_ylabel('Response Time (minutes)')
ax.set_title('Response Time vs Unit Count')
ax.legend()
ax.grid(True, alpha=0.3)

# Coverage
ax = axes[1]
for policy in policies:
    data = results[results['policy_id'] == policy]
    ax.plot(data['K'], data['pct_demand_covered'], 'o-', label=policy, linewidth=2, markersize=8)
ax.set_xlabel('Number of Units (K)')
ax.set_ylabel('% Demand Covered (≤8 min)')
ax.set_title('Coverage vs Unit Count')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Diminishing Returns Analysis

How much improvement do we get from each additional unit?

In [ ]:
# Calculate marginal improvements
print("\nMarginal Improvements (Response Time):")
print("\nP0 (Uniform):")
P0_data = results[results['policy_id'] == 'P0'].sort_values('K')
for i in range(1, len(P0_data)):
    prev = P0_data.iloc[i-1]
    curr = P0_data.iloc[i]
    delta_K = curr['K'] - prev['K']
    delta_RT = prev['response_time'] - curr['response_time']
    print(f"  K={int(prev['K'])}->{int(curr['K'])}: ΔRT = {delta_RT:.2f} min ({delta_RT/delta_K:.3f} min/unit)")

print("\nP2 (Demand-Weighted Optimized):")
P2_data = results[results['policy_id'] == 'P2'].sort_values('K')
for i in range(1, len(P2_data)):
    prev = P2_data.iloc[i-1]
    curr = P2_data.iloc[i]
    delta_K = curr['K'] - prev['K']
    delta_RT = prev['response_time'] - curr['response_time']
    print(f"  K={int(prev['K'])}->{int(curr['K'])}: ΔRT = {delta_RT:.2f} min ({delta_RT/delta_K:.3f} min/unit)")

### Diminishing Returns Insights:

1. **P0 shows strong diminishing returns**:
   - 20->30: 1.16 min/unit improvement
   - 30->40: 0.40 min/unit improvement  
   - 40->48: 0.07 min/unit improvement

2. **P2 achieves near-optimal with K=20**:
   - Only 0.05 min improvement from K=20 to K=30
   - Essentially flat after K=30

3. **Optimal K appears to be ~20-30 units** for optimized policies

## 5. Policy Recommendations

Based on the analysis:

In [ ]:
# Rank policies by average performance
avg_perf = results.groupby('policy_id').agg({
    'response_time': 'mean',
    'pct_demand_covered': 'mean',
    'num_firehouses_used': 'mean'
}).round(2)

avg_perf = avg_perf.sort_values('response_time')
avg_perf['policy_name'] = [results[results['policy_id']==p]['policy_name'].iloc[0] 
                            for p in avg_perf.index]

print("\nAverage Performance Ranking (across all K):")
print(avg_perf[['policy_name', 'response_time', 'pct_demand_covered', 'num_firehouses_used']].to_string())

### Recommendations:

1. **Best overall: P2 (Demand-Weighted)** or **P2b (P-Median)**
   - Identical response times (2.49-2.54 min)
   - 100% coverage at K≥20
   - P2 uses fewer firehouses (more concentrated)

2. **Budget-constrained: Use K=20-30** with P2 policy
   - Achieves 2.54 min response time with just 20 units
   - Diminishing returns beyond K=30

3. **Political feasibility: P1 (Demand-Proportional)**
   - Simple, explainable allocation
   - Only 2-4% worse than optimal
   - Good coverage (100% at K≥20)

4. **Avoid: P0 (Uniform)**
   - 86% worse response time at K=20
   - Only competitive at K≥48 (all firehouses staffed)

5. **Avoid: P2c (Maximal Coverage)**
   - Extreme concentration (7-11 firehouses only)
   - 40% worse response time than P2
   - Only use if coverage is the sole objective

## 6. Maps

Visual comparison of allocation patterns for K=40.

See:
- `results/maps/map_allocation_P0_K40.png` - Uniform allocation
- `results/maps/map_allocation_P1_K40.png` - Demand-proportional
- `results/maps/map_allocation_P2_K40.png` - Optimized allocation

**Key observations**:
- P2 concentrates units in high-demand precincts (Midtown, Financial District)
- P0 spreads units evenly, leaving high-demand areas under-served
- P1 is intermediate, following demand but less concentrated than P2

## Output Files

| File | Description |
|------|-------------|
| `allocations_K*.csv` | Unit allocations for each policy at K ∈ {20,30,40,48} |
| `policy_comparison.csv` | Full comparison table with all metrics |
| `sensitivity_analysis.csv` | Response time and coverage by K and policy |
| `findings_summary.json` | Best policies and key results |
| `fig_policy_comparison.png` | 4-panel visualization |
| `fig_tradeoff_curve.png` | Response time vs coverage scatter |
| `map_allocation_*.png` | Spatial allocation maps |